# Notebook 18: Neural Interventions

## Optogenetics, Pharmacology, and Stimulation for Causal Testing

**Author**: NeuroS Team  
**Date**: 2025-11-25  
**Duration**: ~30 minutes

---

## Overview

Neural interventions are **gold standard tools** for establishing causality in neuroscience. This notebook demonstrates how to model and apply these techniques to foundation models:

### Core Topics:

1. **Optogenetics** - Light-activated ion channels (ChR2, NpHR, ChETA, ReaChR)
2. **Pharmacology** - Drug dose-response curves and receptor dynamics
3. **Electrical Stimulation** - Deep brain stimulation (DBS), transcranial magnetic stimulation (TMS)
4. **Chemogenetics** - Designer receptors (DREADDs)
5. **Foundation Model Applications** - Causal testing of AI representations

### Why Interventions Matter:

**Correlation ≠ Causation**. Observational methods (recording neural activity) can suggest relationships but can't prove causality. Interventions allow us to:

- **Establish causality**: "Does activation of neuron X cause behavior Y?"
- **Test necessity**: "Is circuit A necessary for function B?"
- **Test sufficiency**: "Is activation of pathway C sufficient to drive outcome D?"
- **Dissect mechanisms**: Selectively perturb components to understand interactions

### Key References:

- **Deisseroth (2011)**: Optogenetics - principles and applications
- **Roth (2016)**: DREADDs for remote control of neuronal activity
- **Mayberg et al. (2005)**: DBS for treatment-resistant depression
- **Hallett (2007)**: Transcranial magnetic stimulation: a primer

---

In [ ]:
# Core imports
import numpy as np
import torch
import torch.nn as nn
from typing import List, Tuple, Dict, Callable
import warnings
warnings.filterwarnings('ignore')

# Bokeh for interactive visualizations
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot, column, row
from bokeh.models import HoverTool, Span, Label, ColorBar, LinearColorMapper, BoxAnnotation
from bokeh.palettes import Viridis256, Turbo256, RdYlBu11, Category10
from bokeh.transform import linear_cmap

# NeuroS-MechInt intervention components
from neuros_mechint.interventions import (
    ChR2, NpHR, ChETA, ReaChR,
    OptoStimulator,
    Drugs,
    DBS, TMS, TDCS
)

# Enable Bokeh in Jupyter
output_notebook()

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

print("✓ All imports successful")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

---

## Part 1: Optogenetics

### Background: Light-Activated Ion Channels

**Optogenetics** revolutionized neuroscience by enabling millisecond-precision control of genetically defined neurons using light. Key opsins:

- **ChR2** (Channelrhodopsin-2): Blue light (470 nm) → depolarization (excitation)
- **NpHR** (Halorhodopsin): Yellow light (590 nm) → hyperpolarization (inhibition)
- **ChETA**: Faster variant of ChR2 for high-frequency stimulation
- **ReaChR**: Red-shifted (~590 nm) for deeper tissue penetration

**Physics**: Photocurrent $I_{\text{photo}}$ depends on:
1. Light intensity $\phi$ (mW/mm²)
2. Wavelength $\lambda$ (absorption spectrum)
3. Opsin state dynamics (open/closed/inactivated)

### Channelrhodopsin-2 (ChR2) Response

In [ ]:
# Create ChR2 opsin with realistic parameters
chr2 = ChR2(g_max=1000.0, wavelength_peak=470.0)  # Blue light, 470 nm

# Simulation parameters
dt = 0.1  # ms
duration = 200  # ms
n_steps = int(duration / dt)
time = np.arange(n_steps) * dt

# Light pulse protocol: 10 ms pulse at 10 mW/mm²
light_intensity = np.zeros(n_steps)
light_intensity[500:600] = 10.0  # 50-60 ms

# Simulate membrane response
V = -70.0 * torch.ones(1)  # Resting potential
photocurrents = []
voltages = []

for i, intensity in enumerate(light_intensity):
    I_photo = chr2.photocurrent(V, intensity, dt)
    
    # Simple membrane integration (C_m * dV/dt = I_photo)
    C_m = 1.0  # μF/cm²
    dV = (float(I_photo) / C_m) * dt
    V += dV
    
    photocurrents.append(float(I_photo))
    voltages.append(float(V))

photocurrents = np.array(photocurrents)
voltages = np.array(voltages)

print(f"✓ ChR2 simulation complete")
print(f"  Peak depolarization: {voltages.max() - voltages[0]:.2f} mV")
print(f"  Time to peak: {time[np.argmax(voltages)]:.1f} ms")
print(f"  Peak photocurrent: {photocurrents.max():.1f} pA")

In [ ]:
# Interactive visualization of ChR2 response

# Panel 1: Light stimulus
p_light = figure(
    title='Light Stimulus',
    width=900,
    height=200,
    x_axis_label='Time (ms)',
    y_axis_label='Intensity (mW/mm²)'
)
p_light.line(time, light_intensity, line_width=2, color='blue')
# Add shaded region for light pulse
light_box = BoxAnnotation(left=50, right=60, fill_alpha=0.2, fill_color='cyan')
p_light.add_layout(light_box)
hover_light = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Intensity', '@y{0.0} mW/mm²')])
p_light.add_tools(hover_light)

# Panel 2: Photocurrent
p_current = figure(
    title='ChR2 Photocurrent',
    width=900,
    height=250,
    x_axis_label='Time (ms)',
    y_axis_label='Current (pA)',
    x_range=p_light.x_range
)
p_current.line(time, photocurrents, line_width=2, color='green', legend_label='I_photo')
hover_current = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Current', '@y{0.0} pA')])
p_current.add_tools(hover_current)
p_current.legend.location = 'top_right'

# Panel 3: Membrane voltage
p_voltage = figure(
    title='Membrane Depolarization',
    width=900,
    height=250,
    x_axis_label='Time (ms)',
    y_axis_label='Voltage (mV)',
    x_range=p_light.x_range
)
p_voltage.line(time, voltages, line_width=2, color='red', legend_label='V_m')
# Add resting potential reference
rest_line = Span(location=-70, dimension='width', line_color='gray', line_dash='dashed', line_width=1)
p_voltage.add_layout(rest_line)
hover_voltage = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Voltage', '@y{0.0} mV')])
p_voltage.add_tools(hover_voltage)
p_voltage.legend.location = 'bottom_right'

# Combine panels
grid_chr2 = gridplot([[p_light], [p_current], [p_voltage]])
show(grid_chr2)

print("\n📊 ChR2 Optogenetic Response")
print("   Top: Blue light pulse (470 nm)")
print("   Middle: Inward photocurrent through ChR2")
print("   Bottom: Membrane depolarization")

### Comparing Multiple Opsins

Different opsins have distinct kinetics and spectral properties. Let's compare them:

In [ ]:
# Create multiple opsins
opsins = {
    'ChR2': ChR2(wavelength_peak=470.0),           # Standard, blue
    'ChETA': ChETA(wavelength_peak=470.0),         # Fast, blue
    'ReaChR': ReaChR(wavelength_peak=590.0),       # Red-shifted
    'NpHR': NpHR(wavelength_peak=590.0)            # Inhibitory, yellow
}

# Test with same light protocol
n_steps_opsin = 2000
time_opsin = np.arange(n_steps_opsin) * 0.1
light_pulse = np.zeros(n_steps_opsin)
light_pulse[500:700] = 10.0  # 20 ms pulse

responses = {}
for name, opsin in opsins.items():
    V = -70.0 * torch.ones(1)
    response = []
    
    for intensity in light_pulse:
        I = opsin.photocurrent(V, intensity, 0.1)
        response.append(float(I))
    
    responses[name] = np.array(response)

print("✓ Multi-opsin comparison complete")
for name, resp in responses.items():
    print(f"  {name}: peak = {resp.max():.1f} pA, τ_off = {estimate_tau_off(time_opsin, resp):.1f} ms")

def estimate_tau_off(time, response):
    """Estimate off-kinetics time constant."""
    peak_idx = np.argmax(np.abs(response))
    if peak_idx >= len(response) - 10:
        return 0.0
    
    decay = response[peak_idx:]
    time_decay = time[peak_idx:] - time[peak_idx]
    
    # Find 37% decay point
    target = response[peak_idx] * 0.37
    idx_37 = np.where(np.abs(decay) <= np.abs(target))[0]
    
    if len(idx_37) > 0:
        return time_decay[idx_37[0]]
    return 0.0

In [ ]:
# Interactive comparison of opsins

colors_opsin = {'ChR2': 'blue', 'ChETA': 'cyan', 'ReaChR': 'red', 'NpHR': 'orange'}

# Panel 1: Light stimulus
p_stim = figure(
    title='Light Pulse',
    width=900,
    height=200,
    x_axis_label='Time (ms)',
    y_axis_label='Intensity (mW/mm²)'
)
p_stim.line(time_opsin, light_pulse, line_width=2, color='black')
pulse_box = BoxAnnotation(left=50, right=70, fill_alpha=0.2, fill_color='yellow')
p_stim.add_layout(pulse_box)

# Panel 2: All opsin responses
p_compare = figure(
    title='Opsin Response Comparison',
    width=900,
    height=400,
    x_axis_label='Time (ms)',
    y_axis_label='Photocurrent (pA)',
    x_range=p_stim.x_range
)

for name, response in responses.items():
    p_compare.line(time_opsin, response, line_width=2, 
                   color=colors_opsin[name], legend_label=name, alpha=0.8)

# Add zero reference
zero_line = Span(location=0, dimension='width', line_color='gray', line_dash='dashed', line_width=1)
p_compare.add_layout(zero_line)

hover_compare = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Current', '@y{0.0} pA')])
p_compare.add_tools(hover_compare)
p_compare.legend.location = 'top_right'
p_compare.legend.click_policy = 'hide'

# Panel 3: Kinetics comparison (zoomed on offset)
p_kinetics = figure(
    title='Off-Kinetics Detail',
    width=900,
    height=350,
    x_axis_label='Time (ms)',
    y_axis_label='Normalized Current'
)

# Normalize and align at light offset
offset_time = 70  # ms
offset_idx = int(offset_time / 0.1)
window = 500  # samples

for name, response in responses.items():
    if response[offset_idx] != 0:
        normalized = response[offset_idx:offset_idx+window] / response[offset_idx]
        time_rel = time_opsin[:window]
        p_kinetics.line(time_rel, normalized, line_width=2, 
                       color=colors_opsin[name], legend_label=name, alpha=0.8)

# Add 37% decay reference (tau definition)
tau_line = Span(location=0.37, dimension='width', line_color='red', line_dash='dotted', line_width=2)
p_kinetics.add_layout(tau_line)
tau_label = Label(x=5, y=0.4, text='τ (37%)', text_font_size='10pt', text_color='red')
p_kinetics.add_layout(tau_label)

hover_kin = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Norm. Current', '@y{0.00}')])
p_kinetics.add_tools(hover_kin)
p_kinetics.legend.location = 'top_right'
p_kinetics.legend.click_policy = 'hide'

# Combine
grid_opsin = gridplot([[p_stim], [p_compare], [p_kinetics]])
show(grid_opsin)

print("\n📊 Opsin Comparison")
print("   ChR2: Standard excitatory opsin (blue light)")
print("   ChETA: Faster kinetics for high-frequency stimulation")
print("   ReaChR: Red-shifted for deeper tissue penetration")
print("   NpHR: Inhibitory (hyperpolarizing) opsin")

---

## Part 2: Pharmacology

### Dose-Response Curves

Drugs modulate neural activity by binding to receptors/channels. The **Hill equation** describes dose-response:

$$
E = E_{\max} \frac{[D]^n}{EC_{50}^n + [D]^n}
$$

where:
- $E$ is effect (0-1)
- $[D]$ is drug concentration
- $EC_{50}$ is half-maximal effective concentration
- $n$ is Hill coefficient (cooperativity)

### Common Neuroscience Drugs

In [ ]:
# Create drug models
drugs = {
    'TTX': Drugs.TTX(),         # Tetrodotoxin - Na+ channel blocker
    'TEA': Drugs.TEA(),         # Tetraethylammonium - K+ channel blocker
    'APV': Drugs.APV(),         # NMDA receptor antagonist
    'CNQX': Drugs.CNQX(),       # AMPA receptor antagonist
    'Gabazine': Drugs.Gabazine(),  # GABA_A antagonist
    'Muscimol': Drugs.Muscimol()   # GABA_A agonist
}

# Generate dose-response curves
doses = np.logspace(-3, 2, 100)  # 0.001 to 100 μM

dose_responses = {}
for name, drug in drugs.items():
    responses = [drug.dose_response(d) for d in doses]
    dose_responses[name] = np.array(responses)

print("✓ Dose-response curves generated")
for name, drug in drugs.items():
    print(f"  {name}: EC50 = {drug.EC50:.3f} μM, Hill coeff = {drug.hill_coefficient:.2f}")

In [ ]:
# Interactive dose-response visualization

from bokeh.palettes import Category10_6
colors_drug = Category10_6

# Panel 1: All drug dose-response curves
p_dose = figure(
    title='Dose-Response Curves',
    width=900,
    height=500,
    x_axis_type='log',
    x_axis_label='Concentration (μM)',
    y_axis_label='Normalized Response (0-1)',
    y_range=(0, 1.05)
)

for i, (name, response) in enumerate(dose_responses.items()):
    color = colors_drug[i % len(colors_drug)]
    p_dose.line(doses, response, line_width=2, color=color, 
                legend_label=f'{name} (EC50={drugs[name].EC50:.2f})', alpha=0.8)

# Add 50% response line
half_line = Span(location=0.5, dimension='width', line_color='gray', line_dash='dashed', line_width=1)
p_dose.add_layout(half_line)

hover_dose = HoverTool(tooltips=[('Concentration', '@x{0.000} μM'), ('Response', '@y{0.000}')])
p_dose.add_tools(hover_dose)
p_dose.legend.location = 'bottom_right'
p_dose.legend.click_policy = 'hide'

show(p_dose)

print("\n📊 Pharmacological Dose-Response Curves")
print("   Click legend items to hide/show specific drugs")
print("   Notice different EC50 values (potency) and slopes (cooperativity)")

In [ ]:
# Detailed analysis: TTX (sodium channel blocker)
ttx = drugs['TTX']
ttx_response = dose_responses['TTX']

# Find key concentrations
ec10_idx = np.argmin(np.abs(ttx_response - 0.1))
ec50_idx = np.argmin(np.abs(ttx_response - 0.5))
ec90_idx = np.argmin(np.abs(ttx_response - 0.9))

ec10 = doses[ec10_idx]
ec50 = doses[ec50_idx]
ec90 = doses[ec90_idx]

# Create detailed TTX plot
p_ttx = figure(
    title='TTX (Tetrodotoxin) - Sodium Channel Blocker',
    width=900,
    height=450,
    x_axis_type='log',
    x_axis_label='TTX Concentration (μM)',
    y_axis_label='Na+ Channel Block (fraction)',
    y_range=(0, 1.05)
)

# Data points
p_ttx.circle(doses, ttx_response, size=6, color='blue', alpha=0.6, legend_label='Data')

# Hill fit
hill_fit = ttx.dose_response(doses)
p_ttx.line(doses, hill_fit, line_width=3, color='red', alpha=0.8, legend_label='Hill fit')

# Add EC values
ec10_line = Span(location=ec10, dimension='height', line_color='green', line_dash='dotted', line_width=2)
ec50_line = Span(location=ec50, dimension='height', line_color='orange', line_dash='dashed', line_width=2)
ec90_line = Span(location=ec90, dimension='height', line_color='purple', line_dash='dotted', line_width=2)
p_ttx.add_layout(ec10_line)
p_ttx.add_layout(ec50_line)
p_ttx.add_layout(ec90_line)

# Labels
ec10_label = Label(x=ec10*1.5, y=0.15, text=f'EC10={ec10:.3f}', text_font_size='9pt', text_color='green')
ec50_label = Label(x=ec50*1.5, y=0.55, text=f'EC50={ec50:.3f}', text_font_size='9pt', text_color='orange')
ec90_label = Label(x=ec90*1.5, y=0.92, text=f'EC90={ec90:.3f}', text_font_size='9pt', text_color='purple')
p_ttx.add_layout(ec10_label)
p_ttx.add_layout(ec50_label)
p_ttx.add_layout(ec90_label)

hover_ttx = HoverTool(tooltips=[('[TTX]', '@x{0.0000} μM'), ('Block', '@y{0.000}')])
p_ttx.add_tools(hover_ttx)
p_ttx.legend.location = 'bottom_right'

show(p_ttx)

print(f"\n📊 TTX Dose-Response Analysis")
print(f"   EC10: {ec10:.4f} μM")
print(f"   EC50: {ec50:.4f} μM")
print(f"   EC90: {ec90:.4f} μM")
print(f"   Hill coefficient: {ttx.hill_coefficient:.2f}")
print(f"   \n   TTX blocks voltage-gated sodium channels, preventing action potentials")

---

## Part 3: Electrical Stimulation

### Deep Brain Stimulation (DBS)

DBS delivers high-frequency electrical pulses to deep brain structures. Clinically used for:
- Parkinson's disease (subthalamic nucleus, 130 Hz)
- Essential tremor (thalamus)
- Depression (subcallosal cingulate)

**Key parameters**:
- Frequency: 20-200 Hz (typically 130 Hz for PD)
- Amplitude: 1-5 V
- Pulse width: 60-200 μs

In [ ]:
# Create DBS stimulator (Parkinson's disease parameters)
dbs = DBS(
    frequency=130.0,   # Hz
    amplitude=3.0,     # V
    pulse_width=0.06   # ms (60 μs)
)

# Simulate waveform
dt_dbs = 0.01  # ms (high resolution for narrow pulses)
duration_dbs = 100  # ms
n_steps_dbs = int(duration_dbs / dt_dbs)
time_dbs = np.arange(n_steps_dbs) * dt_dbs

# Generate stimulation current
currents = np.array([dbs.stimulate(t) for t in time_dbs])

print(f"✓ DBS waveform generated")
print(f"  Frequency: {dbs.frequency} Hz")
print(f"  Pulse width: {dbs.pulse_width} ms")
print(f"  Number of pulses: {int(duration_dbs * dbs.frequency / 1000)}")

In [ ]:
# Interactive DBS visualization

# Panel 1: Full waveform
p_dbs_full = figure(
    title=f'DBS Waveform ({dbs.frequency} Hz, {dbs.amplitude} V, {dbs.pulse_width} ms pulse)',
    width=900,
    height=250,
    x_axis_label='Time (ms)',
    y_axis_label='Current (mA)'
)
p_dbs_full.line(time_dbs, currents, line_width=1, color='blue')
hover_dbs = HoverTool(tooltips=[('Time', '@x{0.00} ms'), ('Current', '@y{0.00} mA')])
p_dbs_full.add_tools(hover_dbs)

# Panel 2: Zoomed single pulse
zoom_start = int(10 / dt_dbs)
zoom_end = int(18 / dt_dbs)

p_dbs_zoom = figure(
    title='Single Pulse Detail',
    width=900,
    height=300,
    x_axis_label='Time (ms)',
    y_axis_label='Current (mA)'
)
p_dbs_zoom.line(time_dbs[zoom_start:zoom_end], currents[zoom_start:zoom_end], 
                line_width=2, color='darkblue')
# Annotate pulse width
pulse_box = BoxAnnotation(left=time_dbs[zoom_start+200], 
                         right=time_dbs[zoom_start+200]+dbs.pulse_width,
                         fill_alpha=0.2, fill_color='red')
p_dbs_zoom.add_layout(pulse_box)
pw_label = Label(x=time_dbs[zoom_start+220], y=currents.max()*0.7, 
                text=f'PW={dbs.pulse_width} ms', text_font_size='10pt')
p_dbs_zoom.add_layout(pw_label)

hover_zoom = HoverTool(tooltips=[('Time', '@x{0.000} ms'), ('Current', '@y{0.00} mA')])
p_dbs_zoom.add_tools(hover_zoom)

# Panel 3: Frequency spectrum
from scipy import signal as scipy_signal
freqs, psd = scipy_signal.welch(currents, fs=1/(dt_dbs/1000), nperseg=min(1024, len(currents)))

p_spectrum = figure(
    title='Power Spectrum',
    width=900,
    height=300,
    x_axis_label='Frequency (Hz)',
    y_axis_label='Power (dB)',
    y_axis_type='log',
    x_range=(0, 500)
)
p_spectrum.line(freqs, psd, line_width=2, color='green')

# Mark fundamental frequency
fund_line = Span(location=dbs.frequency, dimension='height', 
                line_color='red', line_dash='dashed', line_width=2)
p_spectrum.add_layout(fund_line)
fund_label = Label(x=dbs.frequency+5, y=psd.max()*0.5, 
                  text=f'f₀={dbs.frequency} Hz', text_font_size='10pt', text_color='red')
p_spectrum.add_layout(fund_label)

hover_spec = HoverTool(tooltips=[('Frequency', '@x{0.0} Hz'), ('Power', '@y{0.0e0}')])
p_spectrum.add_tools(hover_spec)

# Combine
grid_dbs = gridplot([[p_dbs_full], [p_dbs_zoom], [p_spectrum]])
show(grid_dbs)

print("\n📊 Deep Brain Stimulation Analysis")
print("   Top: Complete waveform showing periodic pulses")
print("   Middle: Single pulse detail (60 μs width)")
print("   Bottom: Frequency spectrum (peak at stimulation frequency)")

### Transcranial Magnetic Stimulation (TMS)

TMS uses rapidly changing magnetic fields to induce electric fields in cortex. The induced electric field $\mathbf{E}$ follows from Faraday's law:

$$
\nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}
$$

**Applications**:
- Motor cortex mapping
- Treatment-resistant depression (rTMS)
- Cognitive neuroscience research

In [ ]:
# Create TMS stimulator
tms = TMS(
    intensity=100.0,               # % of motor threshold
    coil_position=(0, 0, 50),      # 50 mm above cortical surface
    coil_orientation=(0, 0, -1)    # Pointing downward
)

# Calculate field at grid of points (cortical surface)
x_range = 50  # mm
resolution = 50
x = np.linspace(-x_range, x_range, resolution)
y = np.linspace(-x_range, x_range, resolution)
X, Y = np.meshgrid(x, y)
Z = np.zeros_like(X)  # Cortical surface at z=0

# Compute field strength at each point
field_strength = np.zeros_like(X)
for i in range(resolution):
    for j in range(resolution):
        point = np.array([X[i, j], Y[i, j], Z[i, j]])
        field_strength[i, j] = tms.calculate_field(point)

print(f"✓ TMS field calculated")
print(f"  Peak field: {field_strength.max():.1f} V/m")
print(f"  Field at center: {field_strength[resolution//2, resolution//2]:.1f} V/m")

In [ ]:
# Interactive TMS field visualization

from bokeh.models import ColorBar, LinearColorMapper
from bokeh.palettes import Turbo256

# Create color mapper
color_mapper = LinearColorMapper(palette=Turbo256, low=0, high=field_strength.max())

# Flatten for plotting
x_flat = X.flatten()
y_flat = Y.flatten()
field_flat = field_strength.flatten()

# Create TMS field plot
p_tms = figure(
    title='TMS-Induced Electric Field Distribution',
    width=700,
    height=700,
    x_axis_label='X Position (mm)',
    y_axis_label='Y Position (mm)',
    match_aspect=True
)

# Create image
p_tms.image(image=[field_strength], x=-x_range, y=-x_range, 
            dw=2*x_range, dh=2*x_range, color_mapper=color_mapper)

# Add coil position marker
p_tms.cross(0, 0, size=20, line_width=3, color='white', legend_label='Coil center')

# Add color bar
color_bar = ColorBar(color_mapper=color_mapper, width=15, location=(0, 0),
                    title='E-field (V/m)')
p_tms.add_layout(color_bar, 'right')

p_tms.legend.location = 'top_right'

show(p_tms)

print("\n📊 TMS Electric Field Map")
print("   Hotspot: Peak field directly under coil")
print("   Field decays with distance from coil")
print("   Neurons with E-field > ~100 V/m are likely activated")

---

## Part 4: Foundation Model Applications

### Causal Testing with Virtual Interventions

How can we apply neuroscience interventions to foundation models?

**Analogy Table**:

| Neuroscience | Foundation Model |
|--------------|------------------|
| Optogenetic activation | Amplify specific units |
| Optogenetic inhibition | Silence specific units |
| Pharmacological blockade | Mask features/connections |
| Lesion | Zero-out layer/module |
| Electrical stimulation | Add noise/activation |

This enables **mechanistic dissection** of model representations.

In [ ]:
# Create simple foundation model for intervention testing
class FoundationModelWithInterventions(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=200, num_layers=3):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(input_dim if i == 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        self.output = nn.Linear(hidden_dim, 10)  # 10-class classification
        self.activations = {}
    
    def forward(self, x, interventions=None):
        """Forward pass with optional interventions.
        
        Args:
            x: Input tensor
            interventions: Dict mapping layer_idx -> intervention function
        """
        if interventions is None:
            interventions = {}
        
        h = x
        for i, layer in enumerate(self.layers):
            h = layer(h)
            h = torch.relu(h)
            
            # Apply intervention if specified
            if i in interventions:
                h = interventions[i](h)
            
            self.activations[f'layer_{i}'] = h.clone()
        
        return self.output(h)

# Instantiate model
model_int = FoundationModelWithInterventions()
model_int.eval()

# Generate test data
batch_size = 50
input_test = torch.randn(batch_size, 100)

print("✓ Foundation model created for intervention testing")
print(f"  Layers: {len(model_int.layers)}")
print(f"  Hidden dim: 200 units per layer")

In [ ]:
# Define intervention functions

def optogenetic_activation(units_to_activate):
    """Simulate ChR2 activation: amplify specific units."""
    def intervention(h):
        h_modified = h.clone()
        h_modified[:, units_to_activate] *= 2.0  # 2x amplification
        return h_modified
    return intervention

def optogenetic_inhibition(units_to_inhibit):
    """Simulate NpHR inhibition: silence specific units."""
    def intervention(h):
        h_modified = h.clone()
        h_modified[:, units_to_inhibit] *= 0.1  # 90% reduction
        return h_modified
    return intervention

def pharmacological_block(units_to_block):
    """Simulate TTX block: completely silence units."""
    def intervention(h):
        h_modified = h.clone()
        h_modified[:, units_to_block] = 0.0  # Complete block
        return h_modified
    return intervention

def electrical_stimulation(noise_level=0.5):
    """Simulate DBS: add noise to all units."""
    def intervention(h):
        noise = torch.randn_like(h) * noise_level
        return h + noise
    return intervention

print("✓ Intervention functions defined")

In [ ]:
# Test different interventions

# Baseline
with torch.no_grad():
    output_baseline = model_int(input_test, interventions=None)
    act_baseline = {k: v.clone() for k, v in model_int.activations.items()}

# Optogenetic activation (layer 1, units 50-70)
with torch.no_grad():
    interventions_opto_act = {1: optogenetic_activation(range(50, 70))}
    output_opto_act = model_int(input_test, interventions=interventions_opto_act)
    act_opto_act = {k: v.clone() for k, v in model_int.activations.items()}

# Optogenetic inhibition (layer 1, units 100-120)
with torch.no_grad():
    interventions_opto_inh = {1: optogenetic_inhibition(range(100, 120))}
    output_opto_inh = model_int(input_test, interventions=interventions_opto_inh)
    act_opto_inh = {k: v.clone() for k, v in model_int.activations.items()}

# Pharmacological block (layer 1, units 150-180)
with torch.no_grad():
    interventions_pharm = {1: pharmacological_block(range(150, 180))}
    output_pharm = model_int(input_test, interventions=interventions_pharm)
    act_pharm = {k: v.clone() for k, v in model_int.activations.items()}

# Electrical stimulation (layer 1, all units)
with torch.no_grad():
    interventions_elec = {1: electrical_stimulation(noise_level=0.3)}
    output_elec = model_int(input_test, interventions=interventions_elec)
    act_elec = {k: v.clone() for k, v in model_int.activations.items()}

print("✓ All interventions applied")

# Compute effects
effects = {
    'Optogenetic Activation': (output_opto_act - output_baseline).abs().mean().item(),
    'Optogenetic Inhibition': (output_opto_inh - output_baseline).abs().mean().item(),
    'Pharmacological Block': (output_pharm - output_baseline).abs().mean().item(),
    'Electrical Stimulation': (output_elec - output_baseline).abs().mean().item()
}

for name, effect in effects.items():
    print(f"  {name}: Δoutput = {effect:.4f}")

In [ ]:
# Interactive visualization of intervention effects

# Prepare activation data for layer 1
conditions = {
    'Baseline': act_baseline['layer_1'],
    'Opto Activation': act_opto_act['layer_1'],
    'Opto Inhibition': act_opto_inh['layer_1'],
    'Pharmacological': act_pharm['layer_1'],
    'Electrical Stim': act_elec['layer_1']
}

# Create heatmaps
plots = []
for name, activations in conditions.items():
    data = activations.numpy().T  # (units, samples)
    
    p = figure(
        title=name,
        width=350,
        height=450,
        x_axis_label='Sample',
        y_axis_label='Unit',
        x_range=(0, batch_size),
        y_range=(0, 200)
    )
    
    # Create color mapper
    vmax = max([cond.max().item() for cond in conditions.values()])
    mapper = LinearColorMapper(palette=Viridis256, low=0, high=vmax)
    
    p.image(image=[data], x=0, y=0, dw=batch_size, dh=200, color_mapper=mapper)
    
    # Highlight intervention zones
    if 'Activation' in name:
        box = BoxAnnotation(bottom=50, top=70, fill_alpha=0.1, fill_color='cyan', line_color='cyan')
        p.add_layout(box)
    elif 'Inhibition' in name:
        box = BoxAnnotation(bottom=100, top=120, fill_alpha=0.1, fill_color='red', line_color='red')
        p.add_layout(box)
    elif 'Pharmacological' in name:
        box = BoxAnnotation(bottom=150, top=180, fill_alpha=0.1, fill_color='orange', line_color='orange')
        p.add_layout(box)
    
    plots.append(p)

# Arrange in grid
grid_int = gridplot([plots[:3], plots[3:]], toolbar_location='right')
show(grid_int)

print("\n📊 Intervention Effects on Layer 1 Activations")
print("   Colored boxes indicate intervention zones")
print("   Compare patterns across conditions to assess causal impact")

In [ ]:
# Quantitative analysis: Effect size across layers

layer_names = ['layer_0', 'layer_1', 'layer_2']
intervention_names = ['Opto Act', 'Opto Inh', 'Pharm', 'Elec']
activation_sets = [
    act_opto_act,
    act_opto_inh,
    act_pharm,
    act_elec
]

# Compute effect sizes (normalized difference from baseline)
effect_matrix = np.zeros((len(intervention_names), len(layer_names)))

for i, act_set in enumerate(activation_sets):
    for j, layer in enumerate(layer_names):
        baseline = act_baseline[layer].mean().item()
        intervention = act_set[layer].mean().item()
        effect_matrix[i, j] = (intervention - baseline) / (baseline + 1e-8)

# Create effect size heatmap
from bokeh.models import FixedTicker

p_effects = figure(
    title='Intervention Effect Sizes Across Layers',
    width=600,
    height=500,
    x_range=layer_names,
    y_range=list(reversed(intervention_names)),
    toolbar_location=None
)

# Flatten for rect glyph
layer_list = []
int_list = []
effect_list = []

for i, int_name in enumerate(intervention_names):
    for j, layer_name in enumerate(layer_names):
        layer_list.append(layer_name)
        int_list.append(int_name)
        effect_list.append(effect_matrix[i, j])

# Color mapper for effects
max_abs_effect = max(abs(min(effect_list)), abs(max(effect_list)))
effect_mapper = LinearColorMapper(palette=RdYlBu11[::-1], 
                                  low=-max_abs_effect, high=max_abs_effect)

p_effects.rect(x=layer_list, y=int_list, width=1, height=1, 
              fill_color={'field': 'effect', 'transform': effect_mapper},
              line_color='white', line_width=2,
              source={'x': layer_list, 'y': int_list, 'effect': effect_list})

# Add text annotations
from bokeh.models import LabelSet, ColumnDataSource
source = ColumnDataSource(data={'x': layer_list, 'y': int_list, 
                                'text': [f'{e:+.2f}' for e in effect_list]})
labels = LabelSet(x='x', y='y', text='text', source=source, 
                 text_align='center', text_baseline='middle', text_font_size='9pt')
p_effects.add_layout(labels)

# Color bar
color_bar = ColorBar(color_mapper=effect_mapper, width=15, location=(0, 0),
                    title='Effect Size')
p_effects.add_layout(color_bar, 'right')

show(p_effects)

print("\n📊 Effect Size Analysis")
print("   Red: Positive effect (increased activation)")
print("   Blue: Negative effect (decreased activation)")
print("   Values show normalized change from baseline")
print("   \n   Notice how effects propagate and transform across layers")

---

## Exercises

1. **Design an optogenetic experiment**: Choose a specific hypothesis about your model (e.g., "Units 20-40 in layer 2 encode feature X"). Design targeted activation/inhibition to test it.

2. **Dose-response for models**: Apply pharmacological interventions at varying strengths (0%, 25%, 50%, 75%, 100% block). Plot how model performance degrades.

3. **TMS-inspired perturbations**: Add spatially localized noise to model activations, mimicking TMS field patterns. How does perturbation location affect outputs?

4. **Combination interventions**: Test interactions by applying multiple interventions simultaneously (e.g., activate one circuit while blocking another).

5. **Frequency-dependent effects**: For DBS-like interventions, modulate inputs at different frequencies (5 Hz, 40 Hz, 130 Hz). Do models show resonance effects?

---

## Summary

This notebook demonstrated:

1. **Optogenetics**: Multiple opsins with realistic kinetics and spectral properties
2. **Pharmacology**: Dose-response curves for common neuroscience drugs
3. **Electrical Stimulation**: DBS and TMS with physical modeling
4. **Foundation Model Applications**: Virtual interventions for causal testing

**Key Takeaways**:

- Interventions establish **causality**, not just correlation
- Different tools have different **spatiotemporal precision**
  - Optogenetics: cell-type specific, millisecond precision
  - Pharmacology: receptor specific, slower (seconds-minutes)
  - Electrical: spatially broad, fast (milliseconds)
- Foundation models can be **causally dissected** using analogous interventions
- Effect propagation across layers reveals **mechanistic interactions**

**Next Steps**:

- **Notebook 19**: Cross-species alignment and comparative neuroscience
- Apply these intervention techniques to your own foundation models
- Combine with other mechanistic tools (circuits, dynamics, information flow)

---

### References

1. Deisseroth, K. (2011). Optogenetics. *Nature Methods*, 8(1), 26-29.

2. Roth, B. L. (2016). DREADDs for neuroscientists. *Neuron*, 89(4), 683-694.

3. Mayberg, H. S., et al. (2005). Deep brain stimulation for treatment-resistant depression. *Neuron*, 45(5), 651-660.

4. Hallett, M. (2007). Transcranial magnetic stimulation: a primer. *Neuron*, 55(2), 187-199.

5. Boyden, E. S., et al. (2005). Millisecond-timescale, genetically targeted optical control of neural activity. *Nature Neuroscience*, 8(9), 1263-1268.

---

*Created with NeuroS-MechInt • Advanced Mechanistic Interpretability for Neuroscience Foundation Models*